# OpticalLayout デモンストレーション

`gtrace.layout.OpticalLayout` は、光学系全体（光学素子・光源・追跡ルール）を表すモデルクラスです。
GUIフロントエンドはこのクラスと1対1対応し、GUIでの編集はこのクラスへの変更として反映されます。

想定するワークフローは2フェーズです:

1. **構築フェーズ**: 通常のPythonコードで光学素子を配置・アラインメントする（逐次追跡や固有モード計算はここで行う）
2. **登録後**: 完成した素子と光源を `OpticalLayout` に登録し、非逐次追跡 (`non_seq_trace`) で可視化・微調整・迷光確認を行う

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # リポジトリのルートを優先

import numpy as np

import gtrace.beam as beam
import gtrace.optcomp as opt
import gtrace.optics.gaussian as gauss
import gtrace.draw.renderer as renderer
from gtrace.layout import OpticalLayout, TraceRules
from gtrace.unit import *

pi = np.pi

## 1. 構築フェーズ

従来通りのPythonコードで、光源ビームと3枚のミラーからなる光学系を構築します。
実際の作業では、ここで逐次追跡（`hitFromHR` など）を使ってアラインメントを追い込みます。

In [2]:
# 光源: ウェスト半径1mmのビームを原点からx方向へ
b0 = beam.GaussianBeam(q0=gauss.Rw2q(np.inf, 1*mm), wl=1064*nm,
                       pos=[0, 0], dirAngle=0, name='b0')

# 3枚のミラー (M1・M2で光路を90度ずつ折り曲げ、M3で逆反射させる)
M1 = opt.Mirror(HRcenter=[0.5, 0.0], normAngleHR=deg2rad(180-45),
                diameter=10*cm, thickness=5*cm, wedgeAngle=deg2rad(0.25),
                inv_ROC_HR=0.0, Refl_HR=0.99, Trans_HR=0.01,
                Refl_AR=500*ppm, Trans_AR=1-500*ppm, n=1.45, name='M1')
M2 = opt.Mirror(HRcenter=[0.5, 0.4], normAngleHR=deg2rad(-90+45),
                diameter=10*cm, thickness=5*cm, wedgeAngle=deg2rad(0.25),
                inv_ROC_HR=1.0/2.0, Refl_HR=0.99, Trans_HR=0.01,
                Refl_AR=500*ppm, Trans_AR=1-500*ppm, n=1.45, name='M2')
M3 = opt.Mirror(HRcenter=[0.9, 0.4], normAngleHR=deg2rad(180),
                diameter=10*cm, thickness=5*cm, wedgeAngle=deg2rad(0.25),
                inv_ROC_HR=1/1.0, Refl_HR=0.9, Trans_HR=0.1,
                Refl_AR=500*ppm, Trans_AR=1-500*ppm, n=1.45, name='M3')

In [3]:
b0.width(3)

(np.float64(0.0014256043491974921), np.float64(0.0014256043491974921))

## 2. OpticalLayout への登録と追跡

完成した素子・光源を `OpticalLayout` に登録します。追跡の振る舞いは `TraceRules` で指定します。

- 素子・光源は**参照で**保持されます（コピーされない）
- 名前はレイアウト内で一意でなければなりません（重複登録は `ValueError`）
- `trace()` は光源をコピーしてから追跡するので、登録した光源オブジェクト自体は変化しません

In [4]:
layout = OpticalLayout(optics=[M1, M2, M3], sources=[b0],
                       rules=TraceRules(order=5, power_threshold=1e-4))

beams = layout.trace()
print(f'追跡されたビームの数: {len(beams)}')
print(f'光源ごとの内訳: {[(k, len(v)) for k, v in layout.beams_by_source.items()]}')
print()
print(f"{'name':>6} {'P [W]':>10} {'stray':>6} {'length [m]':>11}")
for b in beams:
    print(f'{b.name:>6} {b.P:>10.3e} {b.stray_order:>6d} {b.length:>11.4f}')

追跡されたビームの数: 16
光源ごとの内訳: [('b0', 16)]

  name      P [W]  stray  length [m]
    b0  1.000e+00      0      0.5000
 M1:s1  1.000e-02      1      0.0571
 M1:r1  9.900e-01      0      0.4000
 M2:s1  9.900e-03      1      0.0567
 M2:r1  9.801e-01      0      0.4000
 M3:s1  9.801e-02      1      0.0487
 M3:r1  8.821e-01      0      0.4000
 M2:s1  8.821e-03      1      0.0564
 M2:r1  8.733e-01      0      0.4000
 M1:s1  8.733e-03      1      0.0574
 M1:r1  8.645e-01      0      1.0000
 M1:t1  8.728e-03      0      1.0000
 M2:t1  8.816e-03      0      1.0000
 M3:t1  9.796e-02      0      1.0000
 M2:t1  9.895e-03      0      1.0000
 M1:t1  9.995e-03      0      1.0000


## 3. 描画

`layout.draw()` はシーングラフ（`Canvas`）を構築して返します。
ここではDXFに書き出しますが、ブラウザで見るHTMLビューアへの書き出しは 8. を参照してください。

In [5]:
cnv = layout.draw()
renderer.renderDXF(cnv, 'OpticalLayout_demo.dxf')
print('レイヤ:', list(cnv.layers.keys()))
print('OpticalLayout_demo.dxf に書き出しました')

レイヤ: ['main_beam', 'main_beam_width', 'stray_beam', 'stray_beam_width', 'Mirrors', 'text']
OpticalLayout_demo.dxf に書き出しました


## 4. 素子ごとの追跡ルール (per_optic_order)

`TraceRules.per_optic_order` に素子名→内部反射次数の辞書を与えると、その素子だけ内部反射の計算次数を変えられます。
不要な迷光の追跡を素子単位で抑制できます。

In [6]:
lay_full = OpticalLayout(optics=[M1, M2, M3], sources=[b0],
                         rules=TraceRules(order=5, power_threshold=1e-8))
lay_cut = OpticalLayout(optics=[M1, M2, M3], sources=[b0],
                        rules=TraceRules(order=5, power_threshold=1e-8,
                                         per_optic_order={'M1': 0}))
print(f'全素子 order=5:              {len(lay_full.trace())} beams')
print(f"M1のみ order=0 (内部反射なし): {len(lay_cut.trace())} beams")

全素子 order=5:              33 beams
M1のみ order=0 (内部反射なし): 25 beams


## 5. 参照セマンティクス — GUIドラッグの基盤

レイアウトは素子を参照で保持しているので、登録済みミラーの属性を書き換えて `trace()` し直すだけで結果が追従します。
将来のGUIでは「ミラーをドラッグ → `HRcenter` を書き換え → 再追跡 → 再描画」という流れになります。

In [7]:
print(f'移動前: 最初のビームの長さ = {layout.trace()[0].length:.3f} m')

M1.HRcenter = [0.6, 0.0]   # ミラーを10cm移動（GUIのドラッグに相当）
print(f'移動後: 最初のビームの長さ = {layout.trace()[0].length:.3f} m')

M1.HRcenter = [0.5, 0.0]   # 元に戻す
print(f'復帰後: 最初のビームの長さ = {layout.trace()[0].length:.3f} m')

移動前: 最初のビームの長さ = 0.500 m
移動後: 最初のビームの長さ = 0.600 m
復帰後: 最初のビームの長さ = 0.500 m


## 6. レイアウトの保存と読み込み

レイアウト（素子・光源・ルール）はJSONファイルとして保存・復元できます。追跡結果は保存されません（`trace()` で再生成できるため）。

In [8]:
layout.save('OpticalLayout_demo_layout.json')

loaded = OpticalLayout.load('OpticalLayout_demo_layout.json')
print(f'読み込んだレイアウト: 素子 {len(loaded.optics)} 個, 光源 {len(loaded.sources)} 個')
print(f'再追跡: {len(loaded.trace())} beams (元のレイアウトと同数)')

with open('OpticalLayout_demo_layout.json') as f:
    print()
    print('JSONファイルの冒頭:')
    print(f.read()[:400], '...')

読み込んだレイアウト: 素子 3 個, 光源 1 個
再追跡: 16 beams (元のレイアウトと同数)

JSONファイルの冒頭:
{
 "name": "Layout",
 "optics": [
  {
   "type": "Mirror",
   "name": "M1",
   "HRcenter": [
    0.5,
    0.0
   ],
   "normAngleHR": 2.356194490192345,
   "diameter": 0.1,
   "thickness": 0.05,
   "wedgeAngle": 0.004363323129985824,
   "inv_ROC_HR": 0.0,
   "inv_ROC_AR": 0.0,
   "Refl_HR": 0.99,
   "Trans_HR": 0.01,
   "Refl_AR": 0.0005,
   "Trans_AR": 0.9995,
   "n": 1.45,
   "HRtransmissive": f ...


## 7. scene_dict — GUIビューアが読むデータ

`layout.scene_dict()` は、描画図形（canvas）とビームの物理パラメータ（beams）をまとめたJSON互換の辞書を返します。
Stage 1以降のHTML/JSビューアはこのデータを読み、ビーム上の任意点クリックで `q' = q + d` からビーム半径やROCを計算して表示します。

In [9]:
sd = layout.scene_dict()
print('トップレベルのキー:', list(sd.keys()))
print('レイヤ:', [ly['name'] for ly in sd['canvas']['layers']])
print()
print('最初のビームのメタデータ:')
for k, v in sd['beams'][0].items():
    print(f'  {k}: {v}')

トップレベルのキー: ['canvas', 'beams']
レイヤ: ['main_beam', 'main_beam_width', 'stray_beam', 'stray_beam_width', 'Mirrors', 'text']

最初のビームのメタデータ:
  name: b0
  layer: main_beam
  pos: [0.0, 0.0]
  end: [0.5, 0.0]
  dirVect: [1.0, 0.0]
  dirAngle: 0.0
  length: 0.5
  wl: 1.064e-06
  n: 1.0
  P: 1.0
  qx: [0.0, 2.952624674426497]
  qy: [0.0, 2.952624674426497]
  wx: 0.001
  wy: 0.001
  Gouyx: 0.0
  Gouyy: 0.0
  optDist: 0.0
  stray_order: 0


## 8. HTMLビューアへの書き出し

`layout.render_html(filename)` は、描画データとビームの物理パラメータを埋め込んだ**自己完結HTML**を1枚書き出します。
外部ファイル・CDN・サーバに一切依存しないので、ダブルクリックで開けますし、共同研究者にそのまま送れます。
DXFを書き出してCADで開く従来のワークフローを、そのまま置き換えるものです。

ブラウザ上でできること:

- ホイールでカーソル中心ズーム、ドラッグでパン
- ビーム上にカーソルを置くと、**その点での**ビームパラメータ（w, ROC, q, ウェスト径と位置, Rayleigh長, Gouy位相, パワー, 光路長）がライブ表示される
- クリックで読み出しを固定。同じ場所を続けてクリックすると、重なったビーム（同じ光路を逆向きに戻るビームなど）を順に切り替えられる
- レイヤごとの表示/非表示

`layout.show()` は同じHTMLを書き出して既定のブラウザで開きます（ファイル名を省略すると一時ファイルを使います）。

In [10]:
html_file = layout.render_html('OpticalLayout_demo.html', title='OpticalLayout demo')
print(f'{html_file} に書き出しました ({os.path.getsize(html_file)/1024:.0f} kB)')

# 自己完結の確認: 外部を参照する src / href が1つも無いこと
import re
with open(html_file, encoding='utf-8') as f:
    doc = f.read()
print('外部参照:', re.findall(r'(?:src|href)\s*=\s*["\']([^"\']+)', doc) or 'なし')

# ブラウザで開くには:
# layout.show()

OpticalLayout_demo.html に書き出しました (99 kB)
外部参照: なし
